---
title: SWOT High Resolution data search and access
date: 2025-09-11
github: https://github.com/cryointhecloud/CryocloudWebsite
subject: Tutorial
authors:
  - name: Tasha Snow
    affiliations:
      - University of Maryland
      - NASA Goddard Space Flight Center
      - CSM
    email: tsnow03@umd.edu
    orcid: 0000-0001-5697-5470
  - name: Zachary Katz
    affiliations:
      - id: CSM
        institution: Colorado School of Mines
        department: Department of Geophysics
    email: zachary_katz@mines.edu
    corresponding: true
    github: zsk4
license: MIT
---

::::{admonition} Acknowledgments
:class: seealso

We thank NASA PO.DAAC and NSIDC for portions of code and text: \
[*NSIDC/CryoCloud earthaccess tutorial*](https://book.cryointhecloud.com/earthaccess) \
[*PODAAC SWOT Multi Raster tutorial*](https://podaac.github.io/tutorials/notebooks/datasets/SWOT_Raster_Notebook_cloud.html)

::::

# Overview

::::{admonition} What you'll learn
:class: tip

By the end of this tutorial, you will be able to:

- Find and stream **SWOT** data from the cloud directly into memory
- How to inspect basic variables and visualize the data interactively

::::{dropdown} Notes for beginners
:class: hint

“Stream into memory” means you don’t download files permanently—you read them on-the-fly from cloud storage.
::::

::::{dropdown} What you’ll do (quick preview)
- Sign in with **Earthdata Login**  
- Search for **SWOT_L2_HR_Raster_100m_D**  
- Open results with `earthaccess.open(...)` and inspect a few variables
- **Visualize** multiple SWOT rasters across time 
::::

::::{admonition} Prerequisites
:class: important

- An [**Earthdata Login**](https://urs.earthdata.nasa.gov/) (free). If you don’t have one yet, create it first.
::::

::::{admonition} Who is this for?
:class: hint
- **Beginners:** You’ll get a gentle, step-by-step start with cloud data.  
- **Experts:** You can skim to the code and links; advanced **Notes** are flagged along the way.
::::

# Introduction to SWOT

In this tutorial, we will open and visualize data from NASA/CNES’s **Surface Water and Ocean Topography (SWOT)** mission in the cloud. In the next tutorial, we’ll compare SWOT elevations to NASA’s **Ice, Cloud, and Land Elevation Satellite-2 (ICESat-2)** over the **Bach Ice Shelf** (Antarctic Peninsula).

```{figure} https://user-images.githubusercontent.com/51928352/153276603-03d101f6-9acc-4502-8d8b-05d2bb58e75e.png 
:width: 100%
:align: center

SWOT *Credit: [PO.DAAC cookbook](https://github.com/podaac/tutorials)*
```
We use the **SWOT High Resolution (HR) Level-2 Water Mask Raster Image Data Product, Version D** (`SWOT_L2_HR_Raster_D`), which includes improved processing for ice-shelf environments. See [**PO.DAAC’s SWOT mission and data overview**](https://podaac.jpl.nasa.gov/SWOT?tab=mission-objectives&sections=about%2Bdata) for product details, processing notes, and documentation.

## Libraries needed to get started

In [1]:
%matplotlib widget

# For searching and accessing NASA data
import earthaccess

# For reading data, analysis and plotting
import xarray as xr
import hvplot.xarray

# For accessing the time dimension from filenames
from datetime import datetime
import re

# For plotting found datasets
import hvplot
import hvplot.xarray

import pprint  # For nice printing of python objects

## EarthData login

An **Earthdata Login** account is required to access (and in many cases stream) NASA data. If you don’t have one yet, register at <https://urs.earthdata.nasa.gov>. It’s free and quick to set up. We’ll use the `earthaccess` library to authenticate. 

Login requires your Earthdata Login username and password. The `login` method will automatically search for these credentials as environment variables or in a `.netrc` file, and if those aren't available it will prompt you to enter your username and password. We use the prompt strategy here.

::::{dropdown} Saving your credentials
A `.netrc` file is a text file located in our home directory that contains login information for remote machines.  If you don't have a `.netrc` file, `login` will create one for you if you use `persist=True`.

```
earthaccess.login(strategy='interactive', persist=True)
```

BUT make sure you **do not** commit your `.netrc` to your Github repo. This is easy to accidentally do via `git add -A` and would be a major security risk.
::::

In [2]:
auth = earthaccess.login()

# Sanity check so you know that your credentals worked.
assert auth.authenticated, "Earthdata Login failed — please re-try."

## Search for SWOT cloud-native collections

`earthaccess` leverages the Common Metadata Repository (CMR) API to search for collections and granules.  [Earthdata Search](https://search.earthdata.nasa.gov/search) also uses the CMR API.

We can use the `search_datasets` method to search for SWOT collections by setting `keyword="SWOT"`. 

::::{dropdown} Advanced search options
The argument passed to `keyword` can be any string and can include wildcard characters `?` or `*`. To see a full list of search parameters you can type `earthaccess.search_datasets?`.  Using `?` after a python object displays the `docstring` for that object.
::::

A count of the number of data collections (Datasets) found is given.

In [3]:
query = earthaccess.search_datasets(
    keyword="SWOT",
    cloud_hosted = True,
    version = 'D'
)
print (f'{len(query)} datasets found.')

35 datasets found.


We can get a summary of each dataset, which includes links for where to find lengthier descriptions of the data. We look at the first five in the query here.

In [4]:
for collection in query[:5]:
    pprint.pprint(collection.summary(), sort_dicts=True, indent=4)
    print('')  # Add a space between collections for readability

{   'cloud-info': {   'Region': 'us-west-2',
                      'S3BucketAndObjectPrefixNames': [   'podaac-swot-ops-cumulus-protected/SWOT_L2_LR_SSH_D/',
                                                          'podaac-swot-ops-cumulus-public/SWOT_L2_LR_SSH_D/'],
                      'S3CredentialsAPIDocumentationURL': 'https://archive.swot.podaac.earthdata.nasa.gov/s3credentialsREADME',
                      'S3CredentialsAPIEndpoint': 'https://archive.swot.podaac.earthdata.nasa.gov/s3credentials'},
    'concept-id': 'C3233945000-POCLOUD',
    'file-type': "[{'FormatType': 'Native', 'Format': 'netCDF-4'}]",
    'get-data': [   'https://cmr.earthdata.nasa.gov/virtual-directory/collections/C3233945000-POCLOUD',
                    'https://search.earthdata.nasa.gov/search/granules?p=C3233945000-POCLOUD'],
    'short-name': 'SWOT_L2_LR_SSH_D',
    'version': 'D'}

{   'cloud-info': {   'Region': 'us-west-2',
                      'S3BucketAndObjectPrefixNames': [   'podaac-swot-ops

For each collection, `summary` returns a subset of fields from the collection metadata and Unified Metadata Model (UMM) entry.

- `concept-id` is an unique identifier for the collection that is composed of a alphanumeric code and the provider-id for the DAAC.
- `file-type` gives information about the file format of the collection files.
- `get-data` is a collection of URLs that can be used to access data, dataset landing pages, and tools. 
- `short-name` is the name of the dataset that appears on the dataset set landing page. For SWOT, `ShortNames` are generally how different products are referred to.
- `version` is the version of each collection.
 

For _cloud-hosted_ data, there is additional information about the location of the S3 bucket that holds the data and where to get credentials to access the S3 buckets.  In general, you don't need to worry about this information because `earthaccess` handles S3 credentials for you.  Nevertheless it may be useful for troubleshooting. 

```{note}
In Python, all data are represented by _objects_.  These _objects_ contain both data and methods (think functions) that operate on the data.  `earthaccess` includes  `DataCollection` and `DataGranule` objects that contain data about collections and granules returned by `search_datasets` and `search_data` respectively.  If you are familiar with Python, you will see that the data in each `DataCollection` object is organized as a hierarchy of python dictionaries, lists and strings.  So if you know the dictionary key for the metadata entry you want you can get that metadata using standard dictionary methods.  Take a look at https://github.com/nsidc/earthaccess/blob/main/earthaccess/results.py#L80 to learn more.
```

For the SWOT search results the end of the concept-id is `POCLOUD` which means this data is located in the PODAAC cloud. 

For SWOT `short-name` refers to the following products. `D` at the end refers to version D, but you can replace that with `2.0` to get version C.

| ShortName | Product Description (with linked tutorials if available) |
|:-----------:|:---------------------|
| <u>SWOT_L2_HR_Raster_D</u> | [SWOT Level 2 Water Mask Raster Image](https://podaac.github.io/tutorials/notebooks/datasets/SWOT_Raster_Notebook_cloud.html) |
| SWOT_L2_HR_Raster_100m_D | 100m spatial resolution |
| SWOT_L2_HR_Raster_250m_D | 250m spatial resolution |
|||
| <u>SWOT_L2_LR_SSH_D</u> | [SWOT Level 2 KaRIn Low Rate Sea Surface Height](https://podaac.github.io/tutorials/notebooks/datasets/DirectCloud_Access_SWOT_Oceanography.html) |
| SWOT_L2_LR_SSH_BASIC_D | Contain a limited set of variables, aimed at the general user |
| SWOT_L2_LR_SSH_EXPERT_D | Contain all related variables, intended for expert users |
| SWOT_L2_LR_SSH_UNSMOOTH_D | Includes all related variables, on the finer resolution "native" grid, minimal smoothing applied |
| SWOT_L2_LR_SSH_WINDWAVE_D | Wind and wave height data |
|||
| <u>Some others</u> ||
| SWOT_L2_HR_PIXC_D | [SWOT Level 2 Water Mask Pixel Cloud](https://podaac.github.io/tutorials/notebooks/datasets/SWOT_PIXC_Area_localmachine.html) |
| SWOT_L1B_HR_SLC_D | SWOT Level 1B High-Rate Single-look Complex |
| *SWOT_L2_HR_RiverSP_D* | SWOT Level 2 River Single-Pass Vector |
| *SWOT_L2_HR_LakeSP_D* | SWOT Level 2 Lake Single-Pass Vector |
| SWOT_L2_NALT_GDR_2.0 | SWOT Level 2 Nadir Altimeter Geophysical Data Record with Waveforms |

[Hydrology products tutorial](https://podaac.github.io/tutorials/notebooks/datasets/SWOTHR_s3Access.html) \
[Oceanography products tutorial](https://podaac.github.io/tutorials/notebooks/datasets/DirectCloud_Access_SWOT_Oceanography.html)

If you want to see just those short-names so you can paste it into the `earthaccess` data access below, you can use this method:

In [5]:
for collection in query[:5]:
    pprint.pprint(collection.summary()['short-name'], sort_dicts=True, indent=4)

'SWOT_L2_LR_SSH_D'
'SWOT_L2_HR_RiverSP_D'
'SWOT_L2_HR_LakeAvg_D'
'SWOT_L2_HR_LakeSP_D'
'SWOT_L2_HR_LakeSP_obs_D'


## Search SWOT data using spatial and temporal filters 

Once, you have identified the dataset you want to work with, you can use the `search_data` method to search a data set with spatial and temporal filters. Since we are using the SWOT HR Raster 100m product for this tutorial, we'll search for those rasters over the Bach Ice Shelf in Antarctica, for May 1 and June 30, 2025.

Either `concept-id` or `short-name` can be used to search for granules from a particular dataset.  If you use `short-name` you also need to set `version`.  If you use `concept-id`, this is all that is required because `concept-id` is unique. 

The temporal range is identified with standard date strings. Latitude-longitude corners of a bounding box are specified as lower left, upper right.  Polygons and points, as well as shapefiles can also be specified.

This will display the number of granules that match our search. 

In [6]:
# Open SWOT data
latmin,latmax = -72.5,-71.5
lonmin,lonmax = -73.4,-70.5
sbox = (lonmin, latmin, lonmax, latmax)

results = earthaccess.search_data(
    short_name="SWOT_L2_HR_Raster_100m_2.0",
    temporal=("2025-05-01", "2025-06-30"), 
    bounding_box=sbox
)

print(f'{len(results)} total')

4 total


We'll get metadata for these 4 granules and display it.  The rendered metadata shows a download link, granule size and two images of the data.

In [7]:
[display(r) for r in results]

Collection: {'Version': '2.0', 'ShortName': 'SWOT_L2_HR_Raster_100m_2.0'}
Spatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'GPolygons': [{'Boundary': {'Points': [{'Latitude': -72.65181134201607, 'Longitude': -77.21115793488569}, {'Latitude': -73.48198838567808, 'Longitude': -74.49439973835379}, {'Latitude': -72.651117785835, 'Longitude': -71.77246909976662}, {'Latitude': -71.85990590464894, 'Longitude': -74.49665689097986}, {'Latitude': -72.65181134201607, 'Longitude': -77.21115793488569}]}}], 'BoundingRectangles': [{'WestBoundingCoordinate': -77.21115793488569, 'SouthBoundingCoordinate': -73.48198838567808, 'EastBoundingCoordinate': -71.77246909976662, 'NorthBoundingCoordinate': -71.85990590464894}]}, 'Track': {'Cycle': 32, 'Passes': [{'Pass': 115, 'Tiles': ['022L', '023L', '022R', '023R']}]}}}
Temporal coverage: {'RangeDateTime': {'EndingDateTime': '2025-05-02T02:39:55.052Z', 'BeginningDateTime': '2025-05-02T02:39:49.610Z'}}
Size(MB): 24.705557823181152
Data: ['https://archive.swot.podaac.earthdata.nasa.gov/podaac-swot-ops-cumulus-protected/SWOT_L2_HR_Raster_2.0/SWOT_L2_HR_Raster_100m_UTM18C_N_x_x_x_032_115_011F_20250502T023949_20250502T023955_PIC2_01.nc']

Collection: {'Version': '2.0', 'ShortName': 'SWOT_L2_HR_Raster_100m_2.0'}
Spatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'GPolygons': [{'Boundary': {'Points': [{'Latitude': -71.85990590464894, 'Longitude': -74.49665689097986}, {'Latitude': -72.651117785835, 'Longitude': -71.77246909976662}, {'Latitude': -71.78474050946939, 'Longitude': -69.30381320756359}, {'Latitude': -71.03064831507298, 'Longitude': -72.01413677246609}, {'Latitude': -71.85990590464894, 'Longitude': -74.49665689097986}]}}], 'BoundingRectangles': [{'WestBoundingCoordinate': -74.49665689097986, 'SouthBoundingCoordinate': -72.65111778583501, 'EastBoundingCoordinate': -69.30381320756359, 'NorthBoundingCoordinate': -71.03064831507298}]}, 'Track': {'Cycle': 32, 'Passes': [{'Pass': 115, 'Tiles': ['022L', '023L', '024L', '025L', '022R', '023R', '024R', '025R']}]}}}
Temporal coverage: {'RangeDateTime': {'EndingDateTime': '2025-05-02T02:40:15.163Z', 'BeginningDateTime': '2025-05-02T02:39:53.953Z'}}
Size(MB): 104.46696949005127
Data: ['https://archive.swot.podaac.earthdata.nasa.gov/podaac-swot-ops-cumulus-protected/SWOT_L2_HR_Raster_2.0/SWOT_L2_HR_Raster_100m_UTM19D_N_x_x_x_032_115_012F_20250502T023953_20250502T024015_PIC2_01.nc']

Collection: {'Version': '2.0', 'ShortName': 'SWOT_L2_HR_Raster_100m_2.0'}
Spatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'GPolygons': [{'Boundary': {'Points': [{'Latitude': -71.03064831507298, 'Longitude': -72.01413677246609}, {'Latitude': -71.78474050946939, 'Longitude': -69.30381320756359}, {'Latitude': -70.88767233082984, 'Longitude': -67.06251666772303}, {'Latitude': -70.16870951166557, 'Longitude': -69.74325461031368}, {'Latitude': -71.03064831507298, 'Longitude': -72.01413677246609}]}}], 'BoundingRectangles': [{'WestBoundingCoordinate': -72.01413677246609, 'SouthBoundingCoordinate': -71.78474050946939, 'EastBoundingCoordinate': -67.06251666772303, 'NorthBoundingCoordinate': -70.16870951166557}]}, 'Track': {'Cycle': 32, 'Passes': [{'Pass': 115, 'Tiles': ['024L', '025L', '026L', '027L', '024R', '025R', '026R', '027R']}]}}}
Temporal coverage: {'RangeDateTime': {'EndingDateTime': '2025-05-02T02:40:35.279Z', 'BeginningDateTime': '2025-05-02T02:40:14.064Z'}}
Size(MB): 33.755720138549805
Data: ['https://archive.swot.podaac.earthdata.nasa.gov/podaac-swot-ops-cumulus-protected/SWOT_L2_HR_Raster_2.0/SWOT_L2_HR_Raster_100m_UTM19D_N_x_x_x_032_115_013F_20250502T024014_20250502T024035_PIC2_01.nc']

Collection: {'Version': '2.0', 'ShortName': 'SWOT_L2_HR_Raster_100m_2.0'}
Spatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'BoundingRectangles': [{'WestBoundingCoordinate': -180, 'SouthBoundingCoordinate': -90, 'EastBoundingCoordinate': 180, 'NorthBoundingCoordinate': 90}]}, 'Track': {'Cycle': 32, 'Passes': [{'Pass': 138, 'Tiles': ['034L', '035L', '036L', '037L', '034R', '035R', '036R', '037R']}]}}}
Temporal coverage: {'RangeDateTime': {'EndingDateTime': '2025-05-02T22:25:31.272Z', 'BeginningDateTime': '2025-05-02T22:25:10.139Z'}}
Size(MB): 67.47187042236328
Data: ['https://archive.swot.podaac.earthdata.nasa.gov/podaac-swot-ops-cumulus-protected/SWOT_L2_HR_Raster_2.0/SWOT_L2_HR_Raster_100m_UTM01W_N_x_x_x_032_138_018F_20250502T222510_20250502T222531_PIC2_01.nc']

[None, None, None, None]

## Open, load and display data stored on S3

Direct-access to data from an S3 bucket is a two step process.  First, the files are opened using the `open` method.  This first step creates a Python file-like object that is used to load the data in the second step.  

Authentication is required for this step.  The `auth` object created at the start of the notebook is used to provide Earthdata Login authentication and AWS credentials "_behind-the-scenes_". These credentials expire after one hour so the `auth` object must be executed within that time window prior to these next steps. 

```{note}
The `open` step to create a file-like object is required because AWS S3, and other cloud storage systems, use  object storage but most HDF5 libraries work with POSIX-compliant file systems.  POSIX stands for Portable Operating System Interface for Unix and is a set of guidelines that include how to interact with files and file systems.  Linux, Unix, MacOS (which is Unix-like), and Windows are POSIX-compliant. Critically, POSIX-compliant systems allows blocks of bytes or individual bytes to be read from a file. With object storage the whole file has to be read. To get around this limitation, an intermediary is used, in this case `s3fs`.  This intermediary creates a local POSIX-compliant virtual file system.  S3 objects are loaded into this virtual file system so they can be accessed using POSIX-style file functions.
```

In [8]:
rasters = earthaccess.open(results)

QUEUEING TASKS | :   0%|          | 0/4 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/4 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/4 [00:00<?, ?it/s]

After reading the data in, we can open one file at a time. In this example, data are loaded into an `xarray.Dataset`.  Data could be read into `numpy` arrays or a `pandas.Dataframe`.  However, each granule would have to be read using a package that reads HDF5 granules such as `h5py`.  `xarray` does this all _under-the-hood_ in a single line. 

In [9]:
d1 = xr.open_dataset(rasters[0])
d1

<xarray.Dataset> Size: 642MB
Dimensions:                  (x: 1811, y: 1810)
Coordinates:
  * x                        (x) float64 14kB 4.264e+05 4.265e+05 ... 6.074e+05
  * y                        (y) float64 14kB 1.846e+06 1.846e+06 ... 2.027e+06
Data variables: (12/39)
    crs                      object 8B ...
    longitude                (y, x) float64 26MB ...
    latitude                 (y, x) float64 26MB ...
    wse                      (y, x) float32 13MB ...
    wse_qual                 (y, x) float32 13MB ...
    wse_qual_bitwise         (y, x) float64 26MB ...
    ...                       ...
    load_tide_fes            (y, x) float32 13MB ...
    load_tide_got            (y, x) float32 13MB ...
    pole_tide                (y, x) float32 13MB ...
    model_dry_tropo_cor      (y, x) float32 13MB ...
    model_wet_tropo_cor      (y, x) float32 13MB ...
    iono_cor_gim_ka          (y, x) float32 13MB ...
Attributes: (12/49)
    Conventions:                   CF-1.7
    title:                         Level 2 KaRIn High Rate Raster Data Product
    source:                        Ka-band radar interferometer
    history:                       2025-05-06T03:42:07Z : Creation
    platform:                      SWOT
    references:                    V1.3.1
    ...                            ...
    x_min:                         426400.0
    x_max:                         607400.0
    y_min:                         1845700.0
    y_max:                         2026600.0
    institution:                   CNES
    product_version:               01

We can open just that one file, but if we want to work with a large timeseries, it is more likely that we want all 4 datasets in one xarray.Dataset. We can do this in on command called `xarray.open_mfdataset`, but in order to concatenate each dataset by time to add another dimension, we use the `preprocess` function built into xarray to add the time dimension. To execute `preprocess` to add a time dimension, we must first build a function that finds the time dimension from the file name and adds that extra dimension for each SWOT pass we have collected.

In [10]:
earthaccess.results.DataGranule.data_links(results[0], access='direct')

['s3://podaac-swot-ops-cumulus-protected/SWOT_L2_HR_Raster_2.0/SWOT_L2_HR_Raster_100m_UTM18C_N_x_x_x_032_115_011F_20250502T023949_20250502T023955_PIC2_01.nc']

In [ ]:
# Preprocess helper to add a time coordinate from the filename
#    Looks for YYYYMMDDTHHMMSS anywhere in the source path
_TIME_RE = re.compile(r"(\d{8}T\d{6})")

def add_time_from_source(ds: xr.Dataset) -> xr.Dataset:
    src = str(ds.encoding.get("source", ""))  # xarray keeps this
    m = _TIME_RE.search(src)
    if m:
        ts = datetime.strptime(m.group(1), "%Y%m%dT%H%M%S")
        # attach as a proper dimension so open_mfdataset can concat
        ds = ds.expand_dims(time=[ts])
    else:
        # fallback: leave unmodified if no timestamp can be found
        pass
    return ds

Then we can run `xarray.open_mfdataset` with that preprocessing function included. This only *lazy loads* the data meaning we can do operations on the data and metadata but the data aren't actually read into memory yet unless we need them. `ds` is only about 1 Gb right now, but if we ran `ds.compute()` to read all of the variables in, `ds` would be ~25 Gb and potentially crash our memory.

In [12]:
# Open as a multi-file dataset concatenated by time - 30s runtime
ds = xr.open_mfdataset(
    rasters,
    engine="h5netcdf",           # recommended for streamed HDF5/NetCDF via fsspec
    preprocess=add_time_from_source,
    combine="nested",            # adding time during preprocess
    concat_dim="time",
    decode_cf=True,
)
ds

<xarray.Dataset> Size: 13GB
Dimensions:                  (time: 4, y: 5441, x: 2983)
Coordinates:
  * x                        (x) float64 24kB 3.092e+05 3.093e+05 ... 6.074e+05
  * y                        (y) float64 44kB 1.846e+06 1.846e+06 ... 7.439e+06
  * time                     (time) datetime64[ns] 32B 2025-05-02T02:39:49 .....
Data variables: (12/39)
    crs                      (time) object 32B b'1' b'1' b'1' b'1'
    longitude                (time, y, x) float64 519MB dask.array<chunksize=(1, 582, 582), meta=np.ndarray>
    latitude                 (time, y, x) float64 519MB dask.array<chunksize=(1, 582, 582), meta=np.ndarray>
    wse                      (time, y, x) float32 260MB dask.array<chunksize=(1, 873, 873), meta=np.ndarray>
    wse_qual                 (time, y, x) float32 260MB dask.array<chunksize=(1, 1746, 1746), meta=np.ndarray>
    wse_qual_bitwise         (time, y, x) float64 519MB dask.array<chunksize=(1, 873, 873), meta=np.ndarray>
    ...                       ...
    load_tide_fes            (time, y, x) float32 260MB dask.array<chunksize=(1, 873, 873), meta=np.ndarray>
    load_tide_got            (time, y, x) float32 260MB dask.array<chunksize=(1, 873, 873), meta=np.ndarray>
    pole_tide                (time, y, x) float32 260MB dask.array<chunksize=(1, 873, 873), meta=np.ndarray>
    model_dry_tropo_cor      (time, y, x) float32 260MB dask.array<chunksize=(1, 873, 873), meta=np.ndarray>
    model_wet_tropo_cor      (time, y, x) float32 260MB dask.array<chunksize=(1, 873, 873), meta=np.ndarray>
    iono_cor_gim_ka          (time, y, x) float32 260MB dask.array<chunksize=(1, 873, 873), meta=np.ndarray>
Attributes: (12/49)
    Conventions:                   CF-1.7
    title:                         Level 2 KaRIn High Rate Raster Data Product
    source:                        Ka-band radar interferometer
    history:                       2025-05-06T03:42:07Z : Creation
    platform:                      SWOT
    references:                    V1.3.1
    ...                            ...
    x_min:                         426400.0
    x_max:                         607400.0
    y_min:                         1845700.0
    y_max:                         2026600.0
    institution:                   CNES
    product_version:               01

Notice that under dimensions, we now have `time` and it is showing we have 4 time steps, aside from the x and y dimensions. 

In [13]:
ds.time.values

array(['2025-05-02T02:39:49.000000000', '2025-05-02T02:39:53.000000000',
       '2025-05-02T02:40:14.000000000', '2025-05-02T22:25:10.000000000'],
      dtype='datetime64[ns]')

\
Now we can plot all of these time steps with a really nice visualization package called `hvplot`. It give a little time widget on the right that allows you to scroll from one time step to the next. We are plotting the `wse` variable, but this can be switched out for any other variable name. You can move the image, zoom, wheel zoom, save the image, reset the image, and toggle the hover capabilities on the upper right of the image.

In [ ]:
timeplot = ds.wse.hvplot.image(y='y', x='x')
timeplot.opts(width=700, height=500, colorbar=True)